In [1]:
import sys, os

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

import pandas as pd
import shutil
from deltalake import DeltaTable

from src.common.delta_writer import write_delta
from src.common.spark_session import get_spark_session
from src.integration.integrated_taxi_trips import GOLD_PATH, build_integrated_taxi_trips

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

spark = get_spark_session()

Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


26/09/08 23:37:11 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 172.27.138.95 instead (on interface en0)
26/09/08 23:37:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 23:37:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Generate gold deltatable

In [2]:
reset = False

if reset and os.path.exists(GOLD_PATH):
    shutil.rmtree(GOLD_PATH)
    print(f"reset: removed {GOLD_PATH}")

df = build_integrated_taxi_trips(spark)

total = df.count()
with_weather = df.filter(df["weather_temperature"].isNotNull()).count()
with_air_quality = df.filter(df["air_quality_pm25"].isNotNull()).count()
with_pickup_zone = df.filter(df["pickup_zone"].isNotNull()).count()

/Users/thilini/.pyenv/versions/3.12.0/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

In [3]:
import os
import shutil

# Remove the existing Gold table before rebuilding
if os.path.exists(GOLD_PATH):
    shutil.rmtree(GOLD_PATH)
    print(f"Removed existing Gold table: {GOLD_PATH}")


written = write_delta(
    df,
    GOLD_PATH,
    mode="overwrite",
    partition_by=["pickup_year", "pickup_month"],
)

Removed existing Gold table: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/integrated_taxi_trips


26/09/08 23:37:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

### Gold table row count and enrichment coverage statistics

In [4]:
print(f"\n{'=' * 70}")
print(f"integrated_taxi_trips: {written} rows written to {GOLD_PATH}")
print(f"{'=' * 70}")
print(f"  trips total:                {total}")
print(f"  with weather match:         {with_weather} ({100*with_weather/total:.1f}%)")
print(f"  with air-quality match:     {with_air_quality} ({100*with_air_quality/total:.1f}%)")
print(f"  with pickup zone match:     {with_pickup_zone} ({100*with_pickup_zone/total:.1f}%)")


integrated_taxi_trips: 9554776 rows written to /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/data/delta/gold/integrated_taxi_trips
  trips total:                9554776
  with weather match:         9554767 (100.0%)
  with air-quality match:     959344 (10.0%)
  with pickup zone match:     9554776 (100.0%)


9,554,776 taxi trips were written to Gold.

9,554,767 trips received weather information. This is nearly 100%.

959,344 trips received a PM2.5 measurement. This is about 10%, meaning the remaining trips have missing air-quality values.

All 9,554,776 trips received a pickup zone, so coverage is 100%.

### Air Quality Enrichment Coverage by Pickup Borough

In [5]:
from pyspark.sql import functions as F

gold = spark.read.parquet(GOLD_PATH)
gold.groupBy("pickup_borough").agg(
    F.count("*").alias("total_trips"),
    F.sum(F.when(F.col("air_quality_pm25").isNotNull(), 1).otherwise(0)).alias("with_aq"),
).withColumn(
    "match_rate_pct", F.round(100 * F.col("with_aq") / F.col("total_trips"), 1)
).orderBy(F.desc("total_trips")).show(truncate=False)

+--------------+-----------+-------+--------------+
|pickup_borough|total_trips|with_aq|match_rate_pct|
+--------------+-----------+-------+--------------+
|Manhattan     |8556764    |0      |0.0           |
|Queens        |837886     |837166 |99.9          |
|Brooklyn      |97219      |97219  |100.0         |
|Unknown       |31588      |0      |0.0           |
|Bronx         |24959      |24959  |100.0         |
|N/A           |5172       |0      |0.0           |
|EWR           |954        |0      |0.0           |
|Staten Island |234        |0      |0.0           |
+--------------+-----------+-------+--------------+



## Manhattan Air Quality Data Availability

Check whether New York County is present in the Air Quality source to explain Manhattan’s missing PM2.5 coverage.

In [6]:
from deltalake import DeltaTable

aq = DeltaTable('data/delta/silver/air_quality').to_pandas()
print("Distinct county_name values:")
print(sorted(aq['county_name'].dropna().unique()))
print()
print("Does 'New York' appear anywhere (any case/whitespace)?")
print(aq['county_name'].str.strip().str.lower().eq('new york').any())

Distinct county_name values:
['Adair', 'Adams', 'Alachua', 'Alameda', 'Alamosa', 'Albany', 'Albemarle', 'Allegan', 'Allegheny', 'Allen', 'Anchorage', 'Androscoggin', 'Anoka', 'Arapahoe', 'Armstrong', 'Aroostook', 'Ashland', 'Atascosa', 'Athens', 'Atlantic', 'Baldwin', 'Baltimore', 'Baltimore (City)', 'Bartholomew', 'Bay', 'Bayamon', 'Beaver', 'Beaverhead', 'Becker', 'Belknap', 'Bell', 'Belmont', 'Beltrami', 'Bennington', 'Bergen', 'Berks', 'Berkshire', 'Bernalillo', 'Bexar', 'Bibb', 'Billings', 'Blair', 'Blount', 'Blue Earth', 'Bolivar', 'Boulder', 'Boundary', 'Bowie', 'Box Elder', 'Boyd', 'Bradford', 'Brazos', 'Brevard', 'Brewster', 'Bristol', 'Bronx', 'Brooke', 'Brookings', 'Broward', 'Brown', 'Buchanan', 'Buncombe', 'Burke', 'Burleigh', 'Butler', 'Butte', 'Cabell', 'Cache', 'Calaveras', 'Calcasieu', 'Cambria', 'Camden', 'Cameron', 'Campbell', 'Carbon', 'Carlton', 'Carson City', 'Carter', 'Cass', 'Catawba', 'Cecil', 'Cedar', 'Centre', 'Champaign', 'Charles', 'Charleston', 'Chatham', 

In [9]:
from pathlib import Path

zones = pd.read_csv(
    Path(PROJECT_ROOT) / "data" / "raw" / "taxi_zone_lookup" / "taxi_zone_lookup.csv"
)

print(zones["Borough"].dropna().unique())

<ArrowStringArray>
['EWR', 'Queens', 'Bronx', 'Manhattan', 'Staten Island', 'Brooklyn', 'Unknown']
Length: 7, dtype: str


In [10]:
import pandas as pd
from pathlib import Path

raw_aq = pd.read_csv(
    Path(PROJECT_ROOT) / "data/raw/air_quality/air_quality.csv",
    dtype=str
)

manhattan = raw_aq[
    (raw_aq["State Code"].str.zfill(2) == "36")
    & (raw_aq["County Code"].str.zfill(3) == "061")
]

print("Manhattan records:", len(manhattan))
print("County names:", manhattan["County Name"].dropna().unique())

Manhattan records: 0
County names: <ArrowStringArray>
[]
Length: 0, dtype: str


New York County corresponds to Manhattan in the NYC Taxi Zone dataset. Its absence from the raw Air Quality data explains Manhattan’s missing PM2.5 measurements, resulting approximately 10.0% overall air quality coverage.

## Analyse delta results

In [12]:
from pathlib import Path
from deltalake import DeltaTable
from pyspark.sql import functions as F
import pandas as pd
from IPython.display import display, Markdown

raw_weather = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(Path(PROJECT_ROOT) / "data/raw/weather/*.csv"))
)

bronze_weather = spark.read.parquet(
    *DeltaTable("data/delta/bronze/weather").file_uris()
)

silver_weather = spark.read.parquet(
    *DeltaTable("data/delta/silver/weather").file_uris()
)

In [13]:
from pyspark.sql import functions as F
from deltalake import DeltaTable
import pandas as pd

GOLD_PATH = "data/delta/gold/integrated_taxi_trips"
gold = spark.read.parquet(*DeltaTable(GOLD_PATH).file_uris())  # only Delta's ACTIVE files, not stale ones

print("="*70)
print("SECTION 0: Row-count integrity (no trips lost or duplicated)")
print("="*70)
silver_trips = spark.read.parquet(*DeltaTable("data/delta/silver/tripdata").file_uris()).filter(F.col("is_valid"))
print("silver valid trips:", silver_trips.count())
print("gold rows:         ", gold.count())
print("trip_id duplicates in gold:", gold.groupBy("trip_id").count().filter("count > 1").count())

SECTION 0: Row-count integrity (no trips lost or duplicated)
silver valid trips: 9554776
gold rows:          9554776


26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/09/08 23:39:44 WARN RowBasedKeyValueBatch: Calling spill() on

trip_id duplicates in gold: 0


### Confirm Required columns presen

In [14]:
print("="*70)
print("Required columns present, with sane types")
print("="*70)
required = {
    "weather_temperature": "weather conditions",
    "weather_relative_humidity": "weather conditions",
    "weather_precipitation": "weather conditions",
    "weather_wind_speed": "weather conditions",
    "weather_pressure": "weather conditions",
    "air_quality_pm25": "air-quality measurements",
    "pickup_zone": "pickup zone",
    "pickup_borough": "pickup borough",
    "dropoff_zone": "dropoff zone",
    "dropoff_borough": "dropoff borough",
}
missing = [c for c in required if c not in gold.columns]
print("Missing required columns:", missing if missing else "NONE - all present")
gold.select(*required.keys()).printSchema()

Required columns present, with sane types
Missing required columns: NONE - all present
root
 |-- weather_temperature: double (nullable = true)
 |-- weather_relative_humidity: double (nullable = true)
 |-- weather_precipitation: double (nullable = true)
 |-- weather_wind_speed: double (nullable = true)
 |-- weather_pressure: double (nullable = true)
 |-- air_quality_pm25: double (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)



### All columns of integrated_taxi_trips (tridata columns + enriched columns)

In [15]:
from deltalake import DeltaTable
from pyspark.sql import functions as F

GOLD_PATH = "data/delta/gold/integrated_taxi_trips"
gold = spark.read.parquet(*DeltaTable(GOLD_PATH).file_uris())

print(f"Total columns: {len(gold.columns)}\n")
gold.printSchema()

Total columns: 41

root
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_timestamp: timestamp_ntz (nullable = true)
 |-- dropoff_timestamp: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- eve

In [75]:
import pandas as pd

schema_info = [
    {"column": f.name, "type": f.dataType.simpleString(), "nullable": f.nullable}
    for f in gold.schema.fields
]
schema_df = pd.DataFrame(schema_info)
print(schema_df.to_string(index=False))

                   column          type  nullable
               event_date          date      True
               event_hour           int      True
      dropoff_location_id           int      True
       pickup_location_id           int      True
                vendor_id           int      True
         pickup_timestamp timestamp_ntz      True
        dropoff_timestamp timestamp_ntz      True
          passenger_count           int      True
            trip_distance        double      True
              ratecode_id           int      True
       store_and_fwd_flag        string      True
             payment_type           int      True
              fare_amount        double      True
                    extra        double      True
                  mta_tax        double      True
               tip_amount        double      True
             tolls_amount        double      True
    improvement_surcharge        double      True
             total_amount        double      True


### Sample of Selected Columns from the integrated_taxi_trips Delta Table

In [54]:
from deltalake import DeltaTable
from pyspark.sql import functions as F
from IPython.display import display

gold = spark.read.option("basePath", GOLD_PATH).parquet(
    *DeltaTable(GOLD_PATH).file_uris()
)

sample = gold.select(
    "trip_id",
    "pickup_timestamp",
    "weather_temperature",
    "weather_relative_humidity",
    "weather_precipitation",
    "weather_wind_speed",
    "weather_condition_code",
    "air_quality_pm25",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "dropoff_borough",
).dropna().orderBy(F.rand()).limit(10).toPandas()

display(sample)

,trip_id,pickup_timestamp,weather_temperature,weather_relative_humidity,weather_precipitation,weather_wind_speed,weather_condition_code,air_quality_pm25,pickup_zone,pickup_borough,dropoff_zone,dropoff_borough
0,4643ab2de72786964f6edd7a105bffad8fcfe60123a44d...,2024-03-31 02:21:44,10.6,74.0,0.5,16.6,7,7.60,Downtown Brooklyn/MetroTech,Brooklyn,N/A,Unknown
1,ad6652734627dc4d100646defc48e109e0054c4132a3f0...,2024-03-31 18:49:36,15.0,34.0,0.0,24.1,3,5.40,JFK Airport,Queens,Flushing,Queens
2,7571031c804e3f7cbcc768f30004b88e1387203bca8539...,2024-01-01 01:48:45,6.1,53.0,0.0,11.2,4,16.10,Williamsburg (North Side),Brooklyn,East Chelsea,Manhattan
3,b87932df6be1a779d007a13db8f7eb50e46d795d9bfec9...,2024-03-15 20:38:16,13.9,53.0,0.0,24.1,3,5.75,LaGuardia Airport,Queens,Windsor Terrace,Brooklyn
4,c919a2a1c85b8d39c10b187d4c2a62e6c3b10b2b343a8d...,2024-02-24 02:24:13,6.1,73.0,0.0,11.2,3,13.65,Long Island City/Queens Plaza,Queens,Sunnyside,Queens
5,ce6bbfb3f04405626ac41f533d7deb7808a5f77dfc4022...,2024-01-14 16:36:15,3.9,23.0,0.0,55.4,3,3.20,JFK Airport,Queens,JFK Airport,Queens
6,3ddd6cb1991d77d5586ddc5548402306cc2249e51608ce...,2024-03-12 21:29:50,15.0,33.0,0.0,14.8,2,7.85,JFK Airport,Queens,Lincoln Square West,Manhattan
7,c0c03d314ccc670cda2b88a2559fd55bfab476818115f4...,2024-02-27 19:55:48,10.0,80.0,3.0,16.6,8,12.70,JFK Airport,Queens,Williamsburg (North Side),Brooklyn
8,bff8829838ab9c4d59b9692ecbbb3a417928dd3eb7d0dd...,2024-02-05 06:26:53,0.6,47.0,0.0,20.5,2,3.40,JFK Airport,Queens,Times Sq/Theatre District,Manhattan
9,fc5a92d3918c7dfabb5d41c8f2f171393df2409c47b530...,2024-02-20 15:53:47,2.8,32.0,0.0,14.8,2,4.00,JFK Airport,Queens,Midtown Center,Manhattan


### Verification of Delta Table Correctness

In [17]:
import pandas as pd
from IPython.display import display, Markdown
from pyspark.sql import functions as F

def lookup_source_temp(df, date, hour):
    match = df.filter(
        (F.col("year").cast("int") == date.year)
        & (F.col("month").cast("int") == date.month)
        & (F.col("day").cast("int") == date.day)
        & (F.col("hour").cast("int") == int(hour))
    ).select("temp").limit(1).collect()

    return match[0]["temp"] if match else None


def lookup_silver_temp(date, hour):
    match = silver_weather.filter(
        (F.col("event_date") == date)
        & (F.col("event_hour") == int(hour))
    ).select("temperature").limit(1).collect()

    return match[0]["temperature"] if match else None

display(Markdown("## Weather Association — Raw → Bronze → Silver → Gold"))

sample = gold.select(
    "trip_id",
    "pickup_timestamp",
    "event_timestamp",
    "event_date",
    "event_hour",
    "weather_temperature",
).limit(5).toPandas()

results = []

for _, row in sample.iterrows():
    date = row["event_date"]
    hour = row["event_hour"]

    raw_temp = lookup_source_temp(raw_weather, date, hour)
    bronze_temp = lookup_source_temp(bronze_weather, date, hour)
    silver_temp = lookup_silver_temp(date, hour)
    gold_temp = row["weather_temperature"]

    values = [
        pd.to_numeric(v, errors="coerce")
        for v in [raw_temp, bronze_temp, silver_temp, gold_temp]
    ]

    if all(pd.isna(v) for v in values):
        status = "NO VALUE"
    elif any(pd.isna(v) for v in values):
        status = "MISMATCH"
    elif all(abs(v - values[0]) < 1e-6 for v in values):
        status = "OK"
    else:
        status = "MISMATCH"

    results.append({
        "Trip ID": row["trip_id"][:12] + "...",
        "Pickup time": row["pickup_timestamp"],
        "Date": date,
        "Hour": hour,
        "Raw temp (°C)": values[0],
        "Bronze temp (°C)": values[1],
        "Silver temp (°C)": values[2],
        "Gold temp (°C)": values[3],
        "Status": status,
    })

results_df = pd.DataFrame(results)

display(
    results_df.style
    .format({
        "Raw temp (°C)": "{:.1f}",
        "Bronze temp (°C)": "{:.1f}",
        "Silver temp (°C)": "{:.1f}",
        "Gold temp (°C)": "{:.1f}",
    }, na_rep="—")
    .map(
        lambda v: (
            "background-color: #d1fae5; color: #065f46"
            if v == "OK"
            else "background-color: #fee2e2; color: #991b1b"
            if v == "MISMATCH"
            else "background-color: #fef3c7; color: #92400e"
        ),
        subset=["Status"],
    )
    .hide(axis="index")
)

passed = (results_df["Status"] == "OK").sum()

display(Markdown(
    f"**Verification result:** {passed} of {len(results_df)} "
    "sampled trips matched across all four stages."
))

## Weather Association — Raw → Bronze → Silver → Gold

Trip ID,Pickup time,Date,Hour,Raw temp (°C),Bronze temp (°C),Silver temp (°C),Gold temp (°C),Status
00003366ee34...,2024-01-29 14:04:57,2024-01-29,19,3.3,3.3,3.3,3.3,OK
000077448503...,2024-01-08 19:56:54,2024-01-09,0,5.6,5.6,5.6,5.6,OK
000174d633ce...,2024-01-10 11:52:18,2024-01-10,16,9.4,9.4,9.4,9.4,OK
0001ed81ed9b...,2024-01-12 09:11:57,2024-01-12,14,6.1,6.1,6.1,6.1,OK
000233ad76e7...,2024-01-18 11:52:32,2024-01-18,16,-1.1,-1.1,-1.1,-1.1,OK


**Verification result:** 5 of 5 sampled trips matched across all four stages.

In [19]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from deltalake import DeltaTable
from pyspark.sql import functions as F
from src.integration.integrated_taxi_trips import _prepare_air_quality

# Read Raw, Bronze, and Silver.
raw_aq = spark.read.option("header", True).csv(
    str(Path(PROJECT_ROOT) / "data/raw/air_quality/air_quality.csv")
)
bronze_aq = spark.read.parquet(
    *DeltaTable("data/delta/bronze/air_quality").file_uris()
)
silver_aq = spark.read.parquet(
    *DeltaTable("data/delta/silver/air_quality").file_uris()
)

# Prepare the columns needed by the existing aggregation
def normalize_aq(df):
    mapping = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_num",
        "Parameter Code": "parameter_code",
        "County Name": "county_name",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Sample Measurement": "pm25_concentration",
        "Date GMT": "date_gmt",
        "Time GMT": "time_gmt",
    }

    for old, new in mapping.items():
        if old in df.columns and new not in df.columns:
            df = df.withColumnRenamed(old, new)

    if "event_timestamp" not in df.columns:
        df = df.withColumn(
            "event_timestamp",
            F.to_timestamp(F.concat_ws(" ", "date_gmt", "time_gmt"))
        )

    if "event_date" not in df.columns:
        df = df.withColumn("event_date", F.to_date("event_timestamp"))

    if "event_hour" not in df.columns:
        df = df.withColumn("event_hour", F.hour("event_timestamp"))

    return (
        df
        .withColumn("pm25_concentration", F.col("pm25_concentration").cast("double"))
        .withColumn("latitude", F.col("latitude").cast("double"))
        .withColumn("longitude", F.col("longitude").cast("double"))
        .filter(F.col("parameter_code").cast("int") == 88101)
    )

# Select five Gold trips with Air Quality measurements
sample = (
    gold.filter(F.col("air_quality_pm25").isNotNull())
    .select(
        "trip_id", "pickup_timestamp", "event_date", "event_hour",
        "pickup_borough", "air_quality_pm25"
    )
    .limit(5)
)

hours = sample.select("event_date", "event_hour").distinct()
keys = ["pickup_borough", "event_date", "event_hour"]


def aggregate_stage(df, label):
    return (
        _prepare_air_quality(
            normalize_aq(df).join(F.broadcast(hours), ["event_date", "event_hour"])
        )
        .withColumnRenamed("borough", "pickup_borough")
        .select(*keys, F.col("air_quality_pm25").alias(label))
    )

results = sample

for label, df in [
    ("Raw PM2.5", raw_aq),
    ("Bronze PM2.5", bronze_aq),
    ("Silver PM2.5", silver_aq),
]:
    results = results.join(aggregate_stage(df, label), keys, "left")

results_df = results.select(
    "trip_id",
    "pickup_timestamp",
    "pickup_borough",
    "event_date",
    "event_hour",
    F.col("`Raw PM2.5`"),
    F.col("`Bronze PM2.5`"),
    F.col("`Silver PM2.5`"),
    F.col("air_quality_pm25").alias("Gold PM2.5"),
).toPandas()

# Compare values numerically
pm_cols = ["Raw PM2.5", "Bronze PM2.5", "Silver PM2.5", "Gold PM2.5"]
results_df[pm_cols] = results_df[pm_cols].apply(pd.to_numeric, errors="coerce")

def check_status(row):
    values = row[pm_cols]
    if values.isna().any():
        return "MISSING"
    return "OK" if all(abs(v - values.iloc[-1]) < 1e-6 for v in values) else "MISMATCH"

results_df["Status"] = results_df.apply(check_status, axis=1)
results_df["trip_id"] = results_df["trip_id"].str[:12] + "..."

display(Markdown("## Air Quality Association — Raw → Bronze → Silver → Gold"))

display(
    results_df.style
    .format({col: "{:.2f}" for col in pm_cols}, na_rep="—")
    .map(
        lambda v: (
            "background-color: #d1fae5; color: #065f46" if v == "OK"
            else "background-color: #fee2e2; color: #991b1b" if v == "MISMATCH"
            else "background-color: #fef3c7; color: #92400e"
        ),
        subset=["Status"],
    )
    .hide(axis="index")
)

passed = (results_df["Status"] == "OK").sum()
display(Markdown(
    f"**Verification result:** {passed} of {len(results_df)} sampled trips matched."
))

/Users/thilini/.pyenv/versions/3.12.0/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
                                                                                

## Air Quality Association — Raw → Bronze → Silver → Gold

trip_id,pickup_timestamp,pickup_borough,event_date,event_hour,Raw PM2.5,Bronze PM2.5,Silver PM2.5,Gold PM2.5,Status
0008dbc1d958...,2024-01-08 12:52:10,Queens,2024-01-08,17,3.10,3.10,3.10,3.10,OK
000d6a8421a1...,2024-01-29 19:26:13,Queens,2024-01-30,0,3.95,3.95,3.95,3.95,OK
000cdd8cc1d0...,2024-01-29 22:36:33,Queens,2024-01-30,3,4.20,4.20,4.20,4.20,OK
000a4d723d20...,2024-01-02 16:37:25,Queens,2024-01-02,21,6.30,6.30,6.30,6.30,OK
0008c863240a...,2024-01-13 19:16:33,Queens,2024-01-14,0,3.00,3.00,3.00,3.00,OK


**Verification result:** 5 of 5 sampled trips matched.

In [21]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from deltalake import DeltaTable
from pyspark.sql import functions as F

# Read the Taxi Zone Lookup at each stage.
raw_zones = (
    spark.read.option("header", True)
    .csv(str(Path(PROJECT_ROOT) / "data/raw/taxi_zone_lookup/*.csv"))
)

bronze_zones = spark.read.parquet(
    *DeltaTable("data/delta/bronze/taxi_zone_lookup").file_uris()
)

silver_zones = spark.read.parquet(
    *DeltaTable("data/delta/silver/taxi_zone_lookup").file_uris()
)

# Standardize only the lookup columns needed for comparison
def prepare_zones(df, stage):
    names = {
        "LocationID": "location_id",
        "Borough": "borough",
        "Zone": "zone",
    }
    for old, new in names.items():
        if old in df.columns and new not in df.columns:
            df = df.withColumnRenamed(old, new)

    return df.select(
        F.col("location_id").cast("int").alias("location_id"),
        F.col("zone").alias(f"{stage}_zone"),
        F.col("borough").alias(f"{stage}_borough"),
    )

# Five Gold trips with complete pickup and dropoff geography.
sample = (
    gold.dropna(subset=[
        "pickup_zone", "pickup_borough",
        "dropoff_zone", "dropoff_borough"
    ])
    .select(
        "trip_id", "pickup_location_id", "dropoff_location_id",
        "pickup_zone", "pickup_borough",
        "dropoff_zone", "dropoff_borough",
    )
    .limit(5)
)

results = sample

for stage, df in [
    ("Raw", raw_zones),
    ("Bronze", bronze_zones),
    ("Silver", silver_zones),
]:
    lookup = prepare_zones(df, stage)

    for side in ["pickup", "dropoff"]:
        results = results.join(
            lookup.select(
                F.col("location_id").alias(f"{side}_location_id"),
                F.col(f"{stage}_zone").alias(f"{stage}_{side}_zone"),
                F.col(f"{stage}_borough").alias(f"{stage}_{side}_borough"),
            ),
            on=f"{side}_location_id",
            how="left",
        )

results_df = results.toPandas()

# Compare each field across all four stages
fields = [
    "pickup_zone", "pickup_borough",
    "dropoff_zone", "dropoff_borough",
]

def check_status(row):
    for field in fields:
        values = [
            row[f"Raw_{field}"],
            row[f"Bronze_{field}"],
            row[f"Silver_{field}"],
            row[field],
        ]
        if any(pd.isna(v) for v in values):
            return "MISSING"
        if len(set(values)) != 1:
            return "MISMATCH"
    return "OK"

results_df["Status"] = results_df.apply(check_status, axis=1)
results_df["trip_id"] = results_df["trip_id"].str[:12] + "..."

display(Markdown("## Pickup and Dropoff locations — Raw → Bronze → Silver → Gold"))

display_cols = ["trip_id", "pickup_location_id", "dropoff_location_id"]
for field in fields:
    display_cols.extend([
        f"Raw_{field}",
        f"Bronze_{field}",
        f"Silver_{field}",
        field,
    ])
display_cols.append("Status")

display(
    results_df[display_cols].style
    .map(
        lambda v: (
            "background-color: #d1fae5; color: #065f46" if v == "OK"
            else "background-color: #fee2e2; color: #991b1b" if v == "MISMATCH"
            else "background-color: #fef3c7; color: #92400e"
        ),
        subset=["Status"],
    )
    .hide(axis="index")
)

passed = (results_df["Status"] == "OK").sum()
display(Markdown(
    f"**Verification result:** {passed} of {len(results_df)} sampled trips "
    "matched across all four stages."
))

## Pickup and Dropoff locations — Raw → Bronze → Silver → Gold

trip_id,pickup_location_id,dropoff_location_id,Raw_pickup_zone,Bronze_pickup_zone,Silver_pickup_zone,pickup_zone,Raw_pickup_borough,Bronze_pickup_borough,Silver_pickup_borough,pickup_borough,Raw_dropoff_zone,Bronze_dropoff_zone,Silver_dropoff_zone,dropoff_zone,Raw_dropoff_borough,Bronze_dropoff_borough,Silver_dropoff_borough,dropoff_borough,Status
00003366ee34...,239,236,Upper West Side South,Upper West Side South,Upper West Side South,Upper West Side South,Manhattan,Manhattan,Manhattan,Manhattan,Upper East Side North,Upper East Side North,Upper East Side North,Upper East Side North,Manhattan,Manhattan,Manhattan,Manhattan,OK
000077448503...,229,136,Sutton Place/Turtle Bay North,Sutton Place/Turtle Bay North,Sutton Place/Turtle Bay North,Sutton Place/Turtle Bay North,Manhattan,Manhattan,Manhattan,Manhattan,Kingsbridge Heights,Kingsbridge Heights,Kingsbridge Heights,Kingsbridge Heights,Bronx,Bronx,Bronx,Bronx,OK
000174d633ce...,231,79,TriBeCa/Civic Center,TriBeCa/Civic Center,TriBeCa/Civic Center,TriBeCa/Civic Center,Manhattan,Manhattan,Manhattan,Manhattan,East Village,East Village,East Village,East Village,Manhattan,Manhattan,Manhattan,Manhattan,OK
0001ed81ed9b...,50,123,Clinton West,Clinton West,Clinton West,Clinton West,Manhattan,Manhattan,Manhattan,Manhattan,Homecrest,Homecrest,Homecrest,Homecrest,Brooklyn,Brooklyn,Brooklyn,Brooklyn,OK
000233ad76e7...,141,170,Lenox Hill West,Lenox Hill West,Lenox Hill West,Lenox Hill West,Manhattan,Manhattan,Manhattan,Manhattan,Murray Hill,Murray Hill,Murray Hill,Murray Hill,Manhattan,Manhattan,Manhattan,Manhattan,OK


**Verification result:** 5 of 5 sampled trips matched across all four stages.

In [8]:
spark.stop()